In [28]:
# -----######-----###### AIFF ➜ MP3 WITH TAGS + COVER + COMMENT -----######-----######
from pathlib import Path
from mutagen.id3 import ID3, APIC, TIT2, TPE1, TALB, TCON, TDRC, COMM
from mutagen.aiff import AIFF
from mutagen.mp3 import MP3
from PIL import Image
import subprocess
import tempfile

def _aiff_0705_mp3comment_GET_mp3_alltags(path_, use_cbr=True):
    path_ = Path(path_).expanduser()
    if not path_.exists():
        print("❌ File not found."); return

    stem = path_.stem
    parent = path_.parent
    mp3_path = parent / f"{stem}.mp3"

    with tempfile.TemporaryDirectory() as tmpdir:
        tmp_img = Path(tmpdir) / "cover.jpg"

        # Extract cover image
        subprocess.run([
            "ffmpeg", "-y", "-i", str(path_), "-an", "-vcodec", "copy", str(tmp_img)
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        # Convert to MP3
        conv_cmd = [
            "ffmpeg", "-y", "-i", str(path_),
            "-codec:a", "libmp3lame"
        ]
        conv_cmd += ["-b:a", "320k"] if use_cbr else ["-qscale:a", "0"]
        conv_cmd += [str(mp3_path)]
        subprocess.run(conv_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        # Extract metadata
        tags = {}
        try:
            aiff_tags = AIFF(path_)
            tags["title"] = aiff_tags.get("TIT2", "").text[0] if "TIT2" in aiff_tags else ""
            tags["artist"] = aiff_tags.get("TPE1", "").text[0] if "TPE1" in aiff_tags else ""
            tags["album"] = aiff_tags.get("TALB", "").text[0] if "TALB" in aiff_tags else ""
            tags["genre"] = aiff_tags.get("TCON", "").text[0] if "TCON" in aiff_tags else ""
            tags["year"] = aiff_tags.get("TDRC", "").text[0] if "TDRC" in aiff_tags else ""
            tags["comment"] = aiff_tags.get("COMM::eng", "").text[0] if "COMM::eng" in aiff_tags else ""
        except Exception as e:
            print("⚠️ Couldn't extract tags:", e)

        # Apply metadata
        audio = MP3(mp3_path, ID3=ID3)
        try: audio.add_tags()
        except: pass

        if tags.get("title"): audio.tags.add(TIT2(encoding=3, text=tags["title"]))
        if tags.get("artist"): audio.tags.add(TPE1(encoding=3, text=tags["artist"]))
        if tags.get("album"): audio.tags.add(TALB(encoding=3, text=tags["album"]))
        if tags.get("genre"): audio.tags.add(TCON(encoding=3, text=tags["genre"]))
        if tags.get("year"): audio.tags.add(TDRC(encoding=3, text=tags["year"]))
        if tags.get("comment"): audio.tags.add(COMM(encoding=3, lang='eng', desc='', text=tags["comment"]))

        if tmp_img.exists():
            audio.tags.add(APIC(
                encoding=3, mime="image/jpeg", type=3,
                desc="Cover", data=tmp_img.read_bytes()
            ))

        audio.save()
        print(f"✅ MP3 with cover + tags + comment saved: {mp3_path.name}")


In [29]:
path_ = "/Users/yerik/Desktop/STEMS/vocals/10_percent_silence/v1-5700-6BA#maj---25-VARIOUS-007723vocalssil010---sec121-129-luA.aiff"
# HIGH-QUALITY CBR
#_aiff_0705_mp3tags_GET_mp3_cover_tags(path_, use_cbr=True)

_aiff_0705_mp3comment_GET_mp3_alltags(path_, use_cbr=True)



✅ MP3 with cover + tags + comment saved: v1-5700-6BA#maj---25-VARIOUS-007723vocalssil010---sec121-129-luA.mp3
